# `numpy`的批量计算

In [1]:
import os; nb_dir = os.getcwd(); 
import sys; 
sys.path.append(
    os.sep.join([nb_dir, os.pardir, os.pardir, "Customized_Package"])
); 

In [2]:
#以别名形式导入numpy. 
import numpy as np; 
#导入与ndarray有关的数据类型. 
from numpy import int8, int16, int32, int64; 
from numpy import uint8, uint16, uint32, uint64; 
from numpy import float16, float32, float64; 
from numpy import complex64, complex128; 

In [3]:
import ipynb_quick_render

## 基于`ndarray`向量化与广播机制的批量计算

`numpy`中有一类函数, 属于`np.ufinc`对象, 支持向量化运算. 

`np.ufunc`对象分为两类: 一元函数(`nin == 1`)和多元函数(`nin >= 2`). 

In [4]:
#查看numpy模块根上下文中的所有ufunc函数的名称, 功能, 输入输出参数数量
np_ufuncs = [
    (key, val.__doc__.splitlines()[2], val.nin, val.nout) 
    for (key, val) in np.__dict__.items() 
    if type(val) is np.ufunc
]; 
np_ufuncs.sort()

In [5]:
ipynb_quick_render.Table2d(np_ufuncs).render()

_arg,"DO NOT USE, ONLY FOR TESTING",1,1
abs,Calculate the absolute value element-wise.,1,1
absolute,Calculate the absolute value element-wise.,1,1
add,Add arguments element-wise.,2,1
arccos,"Trigonometric inverse cosine, element-wise.",1,1
arccosh,"Inverse hyperbolic cosine, element-wise.",1,1
arcsin,"Inverse sine, element-wise.",1,1
arcsinh,Inverse hyperbolic sine element-wise.,1,1
arctan,"Trigonometric inverse tangent, element-wise.",1,1
arctan2,Element-wise arc tangent of ``x1/x2`` choosing the quadrant correctly.,2,1
arctanh,Inverse hyperbolic tangent element-wise.,1,1


### 一元`ufunc`函数的向量化

一元`np.ufunc`对象作为函数被调用时, 自变量可为标量或`np.array`数组. 
* 自变量为标量(整数, 实数或复数)时, 函数的行为与数学上的函数一致, 也与`math`, `cmath`, `operator`中相同功能的函数一致. 
    * 对于同一自变量和初等函数, `math`(或`cmath`)提供的函数计算结果可能与`numpy`提供的函数计算结果存在微小差异, 主要与不同数值计算算法的截断误差或舍入误差的特征有关. 
* 自变量为`np.array`时, 会返回与原数组的`shape`相同的新数组, 新数组中每个元素, 是原数组中下标相同的元素在该函数下的值. 
$$f\left(\left[
\begin{array}{cc}
 x_{1~1} & x_{1~2} \\ x_{2~1} & x_{2~2} \\
\end{array}
\right]\right) = \left[\begin{array}{cc}
 f\left(x_{1~1}\right) & f\left(x_{1~2}\right) \\
 f\left(x_{2~1}\right) & f\left(x_{2~2}\right) \\
\end{array}\right]$$
    求解过程中, `np.ufunc`会执行向量化过程, 自动遍历原列表中的每个元素, 并在求解后自动存放在新列表中对应的位置. 
* 利用向量化函数自动遍历, 速度比使用循环遍历快(近2个数量级), 因为在RAM寻址上更高效. 

In [6]:
#首项为2, 公比为2^(1/12)的等比数列前373项
geom = np.geomspace(uint32(1), uint32(2) ** 31, 
    31 * 12 + 1, endpoint=True)

In [7]:
#向量化函数的功能与循环遍历后建立数组相同
lin1 = np.array(tuple(np.log2(x) for x in geom)); 
lin2 = np.log2(geom); 
print(np.all(lin1 == lin2))

True


In [8]:
#向量化与循环的速度比较
import math, cmath; 
ln2 = math.log(2); cln2 = cmath.log(2); nln2 = np.log(2); 
%timeit -n 50 -r 50 np.array(tuple(math.log2(x) for x in geom)); 
%timeit -n 50 -r 50 np.array(tuple(math.log(x) / ln2 for x in geom)); 
%timeit -n 50 -r 50 np.array(tuple(cmath.log(x) / cln2 for x in geom)); 
%timeit -n 50 -r 50 np.array(tuple(np.log2(x) for x in geom));  
%timeit -n 50 -r 50 np.array(tuple(np.log(x) / nln2 for x in geom)); 
%timeit -n 50 -r 50 np.log2(geom); 
%timeit -n 50 -r 50 np.log(geom) / nln2; 

162 µs ± 13.8 µs per loop (mean ± std. dev. of 50 runs, 50 loops each)
221 µs ± 41.9 µs per loop (mean ± std. dev. of 50 runs, 50 loops each)
285 µs ± 49.4 µs per loop (mean ± std. dev. of 50 runs, 50 loops each)
1.04 ms ± 136 µs per loop (mean ± std. dev. of 50 runs, 50 loops each)
1.13 ms ± 174 µs per loop (mean ± std. dev. of 50 runs, 50 loops each)
10.5 µs ± 1.64 µs per loop (mean ± std. dev. of 50 runs, 50 loops each)
10.6 µs ± 1.62 µs per loop (mean ± std. dev. of 50 runs, 50 loops each)


### 多元`ufunc`函数的向量化与广播

多元`np.ufunc`对象作为函数被调用时, 自变量可为标量或`np.array`数组. 
* 自变量为标量(整数, 实数或复数)时, 函数的行为与数学上的函数一致, 也与`math`, `cmath`, `operator`中相同功能的函数一致. 
    * 对于同一自变量和初等函数, `math`(或`cmath`)提供的函数计算结果可能与`numpy`提供的函数计算结果存在微小差异, 主要与不同数值计算算法的舍入误差的特征有关. 
* 自变量为`np.array`时: 
    * 如果参与运算的自变量均为同型数组\
        (`shape`属性为`(d_1, d_2, ..., d_n)`)\
        会返回与所有自变量数组的`shape`相同的新数组, 新数组中每个元素, 是自变量数组中下标相同的元素在该函数下的值. 
$$f\left(\left[
\begin{array}{cc}
 x_{1~1} & x_{1~2} \\ x_{2~1} & x_{2~2} \\
\end{array}
\right], \left[
\begin{array}{cc}
 y_{1~1} & y_{1~2} \\ y_{2~1} & y_{2~2} \\
\end{array}\right]
\right) = \left[\begin{array}{cc}
 f\left(x_{1~1}, y_{1~1}\right) & 
 f\left(x_{1~2}, y_{1~2}\right) \\
 f\left(x_{2~1}, y_{2~1}\right) & 
 f\left(x_{2~2}, y_{2~2}\right) \\
\end{array}\right]$$
    求解过程中, `ufunc`会执行向量化过程, 自动遍历原列表中的每个元素, 并在求解后自动存放在新列表中对应的位置. 
    * 如果参与运算的自变量均为同维数组(`ndim`属性为`n`), 但存在两个(含)以上不同型的数组\
        (`shape`属性分别为`(dx_1, dx_2, ..., dx_n)`和`(dy_1, dy_2, ..., dy_n)`), \
        要求**每个`dx_i`($1 \le i \le n$)分别为对应`dy_i`的因数或倍数**, 否则将报错`ValueError`. 
        
        求解过程中, `np.ufunc`首先执行广播过程, 使得所有`np.ndarray`均转化为同型数组后, 方可计算: 
        * 每个`np.ndarray`中元素较少的维度, 将无重叠无分隔地在该维度下"复读", 直到其元素数目与所有`ndarray`中该维度下元素数目最大的数组相同; 
        * 广播过程**不会改变**传入的任何`np.ndarray`的结构. 
        * `np.ndarray`对象的乘法运算(`__mul__`, `__lmul__`和`__imul__`)均被重载为`np.multiply`, 该函数即为`np.ufunc`对象(`nin == 2`), 因此可以表示向量(或矩阵)与数的乘法, 但不能表示向量内积或者矩阵间的乘法. 后者需通过调用`np.dot`函数, `np.array`对象的`dot`方法, 或者`np.linalg.multi_dot`函数实现. 

In [9]:
#二元逻辑运算真值表
logic_operand_1 = np.array([False, False, True, True]); 
logic_operand_2 = np.array([False, True, False, True]); 
logic_oper = (np.logical_and, np.logical_or, np.logical_xor); 
#使用ufunc直接对两个np.array中对应元素批量执行逻辑运算
logic_result = np.array( [
    oper(logic_operand_1, logic_operand_2)
    for oper in logic_oper
] ); 

In [10]:
#构建真值表
truth_table = np.array( [
    [p1, p2] + res.tolist()
    for p1, p2, res 
    in zip(logic_operand_1, logic_operand_2, logic_result.transpose())
] ); 
#显示真值表
truth_table = ipynb_quick_render.Table2d(truth_table); 
truth_table.column_heading = ["A", "B", "A and B", "A or B", "A xor B"]; 
truth_table.render(); 

A,B,A and B,A or B,A xor B
False,False,False,False,False
False,True,False,True,True
True,False,False,True,True
True,True,True,True,False


In [11]:
#多元np.ufunc对同维不同型数组的广播(以两质数相乘为例)
primes_1 = np.array([[2, 3, 5, 7]]); 
primes_2 = primes_1.transpose(); 
ipynb_quick_render.Table2d(primes_1).render(); 
ipynb_quick_render.Table2d(primes_2).render(); 
ipynb_quick_render.Table2d(np.multiply(primes_1, primes_2)).render(); 

2,3,5,7


2
3
5
7


4,6,10,14
6,9,15,21
10,15,25,35
14,21,35,49


## 构造向量化函数
`numpy`中, 使用`np.frompyfunc`可将`def`语句定义的`callable`对象封装为`np.ufunc`对象. 

用法: 
```python
uf = np.frompyfunc(func, nin=n1, nout=n2)
```
* `func` 一般的`python`函数; 
* `n1` 待构造`ufunc`接收的输入参数数量(`nin`属性); 
* `n2` 待构造`ufunc`返回的输出参数数量(`nout`属性)
* 利用该方法构造的`np.ufunc`对`ndarray`批量处理, 返回的新数组的`dtype`始终为`np.object`

In [12]:
#计算自释放起ts末, 自由落体的位移(单位: m)
def free_fall(t): 
    if t <= 0: 
        return 0; 
    else: 
        g = 9.80665; 
        fall = 1 / 2 * g * t ** 2; 
        return fall; 
#函数中t被传入np.ndarray时, 由于定义的函数中t <= 0被自动向量化, 所得结果
#是dtype为np.bool的np.ndarray, 转化为bool时报错. 
try: 
    print(free_fall(np.arange(-5, 5))); 
except Exception as err: 
    print(type(err), str(err)); 

<class 'ValueError'> The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


In [13]:
#使用np.frompyfunc封装为np.ufunc对象后, 当t被传入np.ndarray时, 
#free_fall_batch将以t的元素为单位计算, 不再出现上述异常. 
free_fall_batch = np.frompyfunc(free_fall, 1, 1); 
try: 
    print(free_fall_batch(np.arange(-5, 5)).astype(np.float64)); 
except Exception as err: 
    print(type(err), str(err)); 

[ 0.        0.        0.        0.        0.        0.        4.903325
 19.6133   44.129925 78.4532  ]


In [14]:
#构造向量化函数前后的性能比较
time_phase = np.arange(-5, 5, 0.01); 
%timeit -n 50 -r 50 np.array(tuple(free_fall(t) for t in time_phase)); 
%timeit -n 50 -r 50 np.array(tuple(free_fall_batch(t) for t in time_phase)); 
%timeit -n 50 -r 50 free_fall_batch(time_phase); 

937 µs ± 79.1 µs per loop (mean ± std. dev. of 50 runs, 50 loops each)
4.56 ms ± 497 µs per loop (mean ± std. dev. of 50 runs, 50 loops each)
330 µs ± 35.6 µs per loop (mean ± std. dev. of 50 runs, 50 loops each)
